# 02 — MoleculeNet Benchmarks

Evaluate CAGEFusion on public MoleculeNet datasets using the built-in
`run_moleculenet_benchmark` helper.  No data preparation required — the
helper downloads the dataset, featurizes it, trains the model, and returns
test metrics.

**What you'll learn**
- Run a single benchmark with one function call
- Inspect per-task metrics
- Loop over multiple datasets and build a results table

In [ ]:
# DeepChem is required for MoleculeNet loading
# !pip install deepchem cage_fusion

In [ ]:
from cage_fusion.benchmarks import run_moleculenet_benchmark
import pandas as pd

## Single dataset benchmark

Common datasets: `bace_classification`, `hiv`, `tox21`, `sider`, `clintox`

In [ ]:
results = run_moleculenet_benchmark(
    dataset="bace_classification",
    output_dir="runs/bace",
    num_epochs=50,
    batch_size=128,
    seed=42,
)

print(f"Test ROC-AUC : {results['test_auc']:.4f}")
print(f"Test MCC     : {results['test_mcc']:.4f}")
print(f"Test PR-AUC  : {results['test_pr']:.4f}")
print(f"Checkpoint   : {results['checkpoint_dir']}")

## Per-task breakdown

Multi-task datasets (e.g. Tox21) expose per-task metrics.

In [ ]:
if results["per_task_metrics"]:
    rows = []
    for task, (mcc, auc, pr) in zip(results["label_names"], results["per_task_metrics"]):
        rows.append({"task": task, "AUC": round(auc, 4), "MCC": round(mcc, 4), "PR-AUC": round(pr, 4)})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("Single-task dataset — no per-task breakdown.")

## Training history

In [ ]:
import matplotlib.pyplot as plt

history = results["history"]
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs, history["val_auc"])
axes[1].set_title("Val ROC-AUC")

axes[2].plot(epochs, history["val_mcc"])
axes[2].set_title("Val MCC")

plt.suptitle("bace_classification training history")
plt.tight_layout()
plt.show()

## Multi-dataset loop with seed averaging

In [ ]:
DATASETS = ["bace_classification", "hiv", "tox21"]
SEEDS = [42, 123, 777]

rows = []
for ds in DATASETS:
    aucs, mccs = [], []
    for seed in SEEDS:
        r = run_moleculenet_benchmark(
            dataset=ds,
            output_dir=f"runs/{ds}/seed_{seed}",
            num_epochs=50,
            seed=seed,
        )
        aucs.append(r["test_auc"])
        mccs.append(r["test_mcc"])
    rows.append({
        "dataset": ds,
        "AUC mean": round(sum(aucs) / len(aucs), 4),
        "AUC std": round(pd.Series(aucs).std(), 4),
        "MCC mean": round(sum(mccs) / len(mccs), 4),
        "MCC std": round(pd.Series(mccs).std(), 4),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Custom architecture for benchmarking

Pass a `CageFusionConfig` to override any architectural setting.

In [ ]:
from cage_fusion import CageFusionConfig

# Example: use a larger hidden size and more attention layers
custom_config = CageFusionConfig(
    num_labels=1,            # will be overridden by the data module
    model_task="classification",
    attn_mode="cross",
    co_attention_layers=2,
    hidden_size=256,
)

# NOTE: pass config=None to let run_moleculenet_benchmark set num_labels
# from the data module.  Supply config only to override architecture params.
# The benchmark function always overwrites num_labels and label_names.
print("Custom config ready:", custom_config)